In [1]:
import requests

import matplotlib.pyplot as plt
import seaborn as sns

import umap, numpy as np, pandas as pd

from nltk import FreqDist

import pandas as pd

from sklearn.decomposition import TruncatedSVD
from scipy import sparse
import numpy as np

from collections import Counter
from scipy.sparse import csr_matrix

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd


/home/jupyter-vojta/notebooks/labyrinth/venv_torch_nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-24 16:19:24.549059: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-24 16:19:24.599321: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import warnings
warnings.filterwarnings('ignore')

# Or for specific warning types:
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
labyrinthus_df = pd.read_parquet("../data/large_files/labyrinthus_embeddings.parquet")

In [4]:
labyrinthus_df.columns

Index(['author', 'title', 'grela_id', 'sentence_id', 'sentence_text',
       'context_3sents', 'tokens', 'concordance_tokens', 'not_before',
       'not_after', 'date_random', 'lagt_genre', 'lagt_provenience',
       'noscemus_genre', 'noscemus_discipline', 'target_token_id',
       'target_char_start', 'target_char_end', 'concordance_tokens_str',
       'classification_single', 'classification_single_label', 'xlmr_sentence',
       'xlmr_sp_tokens', 'xlmr_aug_tokens', 'xlmr_conc_text',
       'xlmr_conc_sp_tokens', 'xlmr_conc_aug_tokens', 'labert_sentence',
       'labert_sp_tokens', 'labert_aug_tokens', 'labert_conc_text',
       'labert_conc_sp_tokens', 'labert_conc_aug_tokens', 'cooc_vector',
       'cooc_conc_vector', 'svd400', 'svd400_conc'],
      dtype='object')

## Supervised classification

In [5]:
baseline_embeddings = ['cooc_vector', 'cooc_conc_vector', 'svd400', 'svd400_conc']

xlmr_sentence_contextual_embeddings = [
    'embed_l7_xlmr', 'embed_l8_xlmr', 'embed_l9_xlmr',
    'embed_l10_xlmr', 'embed_l11_xlmr'
]

xlmr_concordance_contextual_embeddings = [
    'embed_conc_l7_xlmr', 'embed_conc_l8_xlmr', 'embed_conc_l9_xlmr',
    'embed_conc_l10_xlmr', 'embed_conc_l11_xlmr'
]


xlmr_sentence_attention_raw_embeddings = [
    'att_l7_xlmr_raw_embed', 'att_l8_xlmr_raw_embed',
    'att_l9_xlmr_raw_embed', 'att_l10_xlmr_raw_embed',
    'att_l11_xlmr_raw_embed'
]

xlmr_sentence_attention_svd400_embeddings = [
    'att_l7_xlmr_svd400_embed', 'att_l8_xlmr_svd400_embed',
    'att_l9_xlmr_svd400_embed', 'att_l10_xlmr_svd400_embed',
    'att_l11_xlmr_svd400_embed'
]

xlmr_concordance_attention_raw_embeddings = [
    'att_conc_l7_xlmr_raw_embed', 'att_conc_l8_xlmr_raw_embed',
    'att_conc_l9_xlmr_raw_embed', 'att_conc_l10_xlmr_raw_embed',
    'att_conc_l11_xlmr_raw_embed']

xlmr_concordance_attention_svd400_embeddings = [
    'att_conc_l7_xlmr_svd400_embed', 'att_conc_l8_xlmr_svd400_embed',
    'att_conc_l9_xlmr_svd400_embed', 'att_conc_l10_xlmr_svd400_embed',
    'att_conc_l11_xlmr_svd400_embed'
]
labert_sentence_contextual_embeddings = [
    'embed_l7_labert', 'embed_l8_labert', 'embed_l9_labert',
    'embed_l10_labert', 'embed_l11_labert'
]

labert_concordance_contextual_embeddings = [
    'embed_conc_l7_labert', 'embed_conc_l8_labert', 'embed_conc_l9_labert',
    'embed_conc_l10_labert', 'embed_conc_l11_labert'
]

labert_sentence_attention_svd400_embeddings = [
    'att_l7_labert_svd400_embed', 'att_l8_labert_svd400_embed',
    'att_l9_labert_svd400_embed', 'att_l10_labert_svd400_embed',
    'att_l11_labert_svd400_embed'
]

labert_sentence_attention_raw_embeddings = [
    'att_l7_labert_raw_embed', 'att_l8_labert_raw_embed',
    'att_l9_labert_raw_embed', 'att_l10_labert_raw_embed',
    'att_l11_labert_raw_embed'
]

labert_concordance_attention_svd400_embeddings = [
    'att_conc_l7_labert_svd400_embed', 'att_conc_l8_labert_svd400_embed',
    'att_conc_l9_labert_svd400_embed', 'att_conc_l10_labert_svd400_embed',
    'att_conc_l11_labert_svd400_embed'
]

labert_concordance_attention_raw_embeddings = [
    'att_conc_l7_labert_raw_embed', 'att_conc_l8_labert_raw_embed',
    'att_conc_l9_labert_raw_embed', 'att_conc_l10_labert_raw_embed',
    'att_conc_l11_labert_raw_embed'
]


In [8]:
def load_embedding(name, base_dir="../data/labyrinth_embeddings"):
    return np.load(f"{base_dir}/{name}.npy", mmap_mode="r")

In [9]:
# ------------------------------------------------------------------
# common objects
# ------------------------------------------------------------------
y = labyrinthus_df["classification_single_label"].values  # 21 classes incl. “other”
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# basic experiments with contextual embeddings

lr = LogisticRegression(
    max_iter=4000,
    solver="lbfgs",
    n_jobs=-1)

rf = RandomForestClassifier(
    n_estimators=200,  # more trees for stability
    max_depth=None,  # let it grow fully
    n_jobs=-1,  # parallelizex
    random_state=42)

hgbc = HistGradientBoostingClassifier(
    random_state=42)

from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(512, 128),
                    activation='relu',
                    solver='adam',
                    alpha=1e-4,
                    max_iter=500,
                    random_state=42)

clfs = {"lr" : lr, "rf" : rf, "hgbc" : hgbc, "mlp" : mlp}

In [12]:
emb_col = "cooc_vector"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6255458515283843, 'F1 (macro)': 0.595166494304927}
{'classifier': 'rf', 'accuracy': 0.618995633187773, 'F1 (macro)': 0.6007395749590153}
{'classifier': 'hgbc', 'accuracy': 0.48580786026200873, 'F1 (macro)': 0.46003800737092687}
{'classifier': 'mlp', 'accuracy': 0.6277292576419214, 'F1 (macro)': 0.6117190524975867}


In [13]:
emb_col = "cooc_conc_vector"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6157205240174672, 'F1 (macro)': 0.6091519636677588}
{'classifier': 'rf', 'accuracy': 0.6026200873362445, 'F1 (macro)': 0.5971272299572506}
{'classifier': 'hgbc', 'accuracy': 0.4497816593886463, 'F1 (macro)': 0.354929068966211}
{'classifier': 'mlp', 'accuracy': 0.6069868995633187, 'F1 (macro)': 0.6091130110235112}


In [14]:
emb_col = "svd400"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6168122270742358, 'F1 (macro)': 0.6010676507591941}
{'classifier': 'rf', 'accuracy': 0.5829694323144105, 'F1 (macro)': 0.5419216936460011}
{'classifier': 'hgbc', 'accuracy': 0.5982532751091703, 'F1 (macro)': 0.5797116507999515}
{'classifier': 'mlp', 'accuracy': 0.5906113537117904, 'F1 (macro)': 0.5732554930348904}


In [15]:
emb_col = "svd400_conc"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6277292576419214, 'F1 (macro)': 0.6157087272609122}
{'classifier': 'rf', 'accuracy': 0.5644104803493449, 'F1 (macro)': 0.46453354210422654}
{'classifier': 'hgbc', 'accuracy': 0.584061135371179, 'F1 (macro)': 0.5593664597099063}
{'classifier': 'mlp', 'accuracy': 0.574235807860262, 'F1 (macro)': 0.5697303913778566}


In [21]:
emb_col = "embed_l9_labert"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.5578602620087336, 'F1 (macro)': 0.5328466858240865}
{'classifier': 'rf', 'accuracy': 0.4978165938864629, 'F1 (macro)': 0.43780492087673206}
{'classifier': 'hgbc', 'accuracy': 0.5185589519650655, 'F1 (macro)': 0.4744671293235527}
{'classifier': 'mlp', 'accuracy': 0.5709606986899564, 'F1 (macro)': 0.5466005598772579}


In [22]:
emb_col = "embed_l11_xlmr"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6451965065502183, 'F1 (macro)': 0.6017240006727936}
{'classifier': 'rf', 'accuracy': 0.5851528384279476, 'F1 (macro)': 0.505609423278255}
{'classifier': 'hgbc', 'accuracy': 0.618995633187773, 'F1 (macro)': 0.5639291096576516}
{'classifier': 'mlp', 'accuracy': 0.6572052401746725, 'F1 (macro)': 0.6206494469510492}


In [25]:
emb_col = "embed_l11_labert"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier": clf_name, "accuracy": acc, "F1 (macro)": f1})

{'classifier': 'lr', 'accuracy': 0.6135371179039302, 'F1 (macro)': 0.5982996892631008}
{'classifier': 'rf', 'accuracy': 0.5, 'F1 (macro)': 0.42522507381019353}
{'classifier': 'hgbc', 'accuracy': 0.5655021834061136, 'F1 (macro)': 0.5288940756331877}
{'classifier': 'mlp', 'accuracy': 0.6157205240174672, 'F1 (macro)': 0.6022711903830064}


In [23]:
emb_col = "embed_l11_labert_wsd"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier": clf_name, "accuracy": acc, "F1 (macro)": f1})

{'classifier': 'lr', 'accuracy': 0.6233624454148472, 'F1 (macro)': 0.6108682807287631}
{'classifier': 'rf', 'accuracy': 0.5218340611353712, 'F1 (macro)': 0.46478299627400316}
{'classifier': 'hgbc', 'accuracy': 0.5567685589519651, 'F1 (macro)': 0.5265439587155128}
{'classifier': 'mlp', 'accuracy': 0.6146288209606987, 'F1 (macro)': 0.6056904640457484}


In [16]:
emb_col = "att_conc_l11_labert_raw_embed"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.3067685589519651, 'F1 (macro)': 0.09737904990559078}
{'classifier': 'rf', 'accuracy': 0.5545851528384279, 'F1 (macro)': 0.539010945729734}
{'classifier': 'hgbc', 'accuracy': 0.4203056768558952, 'F1 (macro)': 0.33046459578199483}
{'classifier': 'mlp', 'accuracy': 0.5633187772925764, 'F1 (macro)': 0.5646092054258923}


In [17]:
emb_col = "att_conc_l11_xlmr_raw_embed"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.29475982532751094, 'F1 (macro)': 0.07588532883642496}
{'classifier': 'rf', 'accuracy': 0.574235807860262, 'F1 (macro)': 0.553441926758296}
{'classifier': 'hgbc', 'accuracy': 0.4366812227074236, 'F1 (macro)': 0.3514466774177026}
{'classifier': 'mlp', 'accuracy': 0.5786026200873362, 'F1 (macro)': 0.5767707195791365}


In [18]:
emb_col = "att_conc_l11_xlmr_svd400_embed"
X = np.stack(load_embedding(emb_col))
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.29475982532751094, 'F1 (macro)': 0.07588532883642496}
{'classifier': 'rf', 'accuracy': 0.5087336244541485, 'F1 (macro)': 0.41309696446330424}
{'classifier': 'hgbc', 'accuracy': 0.5360262008733624, 'F1 (macro)': 0.48321426107514376}
{'classifier': 'mlp', 'accuracy': 0.5502183406113537, 'F1 (macro)': 0.518930443833895}


In [31]:
# appending basic experiments with concatenation of contextual and attention embeddings


def merge_embeddings(embs, method="mean", weights=None) -> np.ndarray:
    """Merge a list of embedding matrices [n, d] via concat / mean / weighted."""
    mats = [np.asarray(load_embedding(e)) for e in embs]
    # sanity checks
    n_rows = [m.shape[0] for m in mats]
    if len(set(n_rows)) != 1:
        raise ValueError(f"Inconsistent n_samples across embeddings: {n_rows}")
    n, d_list = n_rows[0], [m.shape[1] for m in mats]

    if method == "concat":
        return np.concatenate(mats, axis=1)  # [n, sum(d)]
    elif method == "mean":
        M = np.stack(mats, axis=0)           # [k, n, d]
        return M.mean(axis=0)                # [n, d]
    elif method == "weighted":
        if weights is None:
            raise ValueError("weights must be provided for method='weighted'")
        w = np.asarray(weights, dtype=float)
        if w.shape[0] != len(mats):
            raise ValueError(f"weights length {w.shape[0]} != #embs {len(mats)}")
        w = w / w.sum()
        M = np.stack(mats, axis=0)           # [k, n, d]
        return (w[:, None, None] * M).sum(axis=0)  # [n, d]
    else:
        raise ValueError(f"Unknown method: {method}")

In [20]:
L = 11
model = "xlmr"
emb1 = f"embed_l{L}_{model}"
emb2 = f"att_l{L}_{model}_raw_embed"
X = merge_embeddings([emb1, emb2], method="concat")
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6419213973799127, 'F1 (macro)': 0.5985999073911783}
{'classifier': 'rf', 'accuracy': 0.5949781659388647, 'F1 (macro)': 0.5092420827084295}
{'classifier': 'hgbc', 'accuracy': 0.6430131004366813, 'F1 (macro)': 0.6037448172694393}
{'classifier': 'mlp', 'accuracy': 0.6473799126637555, 'F1 (macro)': 0.6138539522857909}


In [24]:
L = 7
model = "xlmr"
emb1 = f"embed_l{L}_{model}"
emb2 = f"att_l{L}_{model}_raw_embed"
X = merge_embeddings([emb1, emb2], method="concat")
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6473799126637555, 'F1 (macro)': 0.6193813168355827}
{'classifier': 'rf', 'accuracy': 0.5873362445414847, 'F1 (macro)': 0.49046015020389727}
{'classifier': 'hgbc', 'accuracy': 0.6408296943231441, 'F1 (macro)': 0.5997427922635491}
{'classifier': 'mlp', 'accuracy': 0.6419213973799127, 'F1 (macro)': 0.6191181231499313}


In [26]:
L = 11
model = "labert"
emb1 = f"embed_l{L}_{model}"
emb2 = f"att_l{L}_{model}_raw_embed"
X = merge_embeddings([emb1, emb2])
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6146288209606987, 'F1 (macro)': 0.5989233881939079}
{'classifier': 'rf', 'accuracy': 0.5491266375545851, 'F1 (macro)': 0.4844848320899091}
{'classifier': 'hgbc', 'accuracy': 0.6048034934497817, 'F1 (macro)': 0.5899326750300419}
{'classifier': 'mlp', 'accuracy': 0.6091703056768559, 'F1 (macro)': 0.589318553018919}


In [27]:
L = 11
model = "labert_wsd"
emb1 = f"embed_l{L}_{model}"
emb2 = f"att_l{L}_{model}_raw_embed"
X = merge_embeddings([emb1, emb2])
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6233624454148472, 'F1 (macro)': 0.6108682807287631}
{'classifier': 'rf', 'accuracy': 0.5403930131004366, 'F1 (macro)': 0.466322014865058}
{'classifier': 'hgbc', 'accuracy': 0.6091703056768559, 'F1 (macro)': 0.5890842199411596}
{'classifier': 'mlp', 'accuracy': 0.6168122270742358, 'F1 (macro)': 0.603843950146945}


In [28]:
L = 11
model = "labert_wsd"
emb1 = f"embed_l{L}_{model}"
emb2 = f"att_l{L}_{model}_raw_embed"
emb3 = f"cooc_vector"
X = merge_embeddings([emb1, emb2, emb3])
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.6561135371179039, 'F1 (macro)': 0.6340670337007888}
{'classifier': 'rf', 'accuracy': 0.5480349344978166, 'F1 (macro)': 0.4728591838067276}
{'classifier': 'hgbc', 'accuracy': 0.6091703056768559, 'F1 (macro)': 0.5890842199411596}
{'classifier': 'mlp', 'accuracy': 0.6790393013100436, 'F1 (macro)': 0.661542042783002}


In [32]:
model = "labert_wsd"
embs = [f"embed_l{L}_{model}" for L in range(9, 12)]
X = merge_embeddings(embs, method="mean")
for clf_name, clf in clfs.items():
    y_pred = cross_val_predict(clf, X, y, cv=cv)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average="macro")
    print({"classifier" : clf_name, "accuracy" : acc, "F1 (macro)" : f1})

{'classifier': 'lr', 'accuracy': 0.5818777292576419, 'F1 (macro)': 0.5656109241705557}
{'classifier': 'rf', 'accuracy': 0.4923580786026201, 'F1 (macro)': 0.4315965477285226}
{'classifier': 'hgbc', 'accuracy': 0.5698689956331878, 'F1 (macro)': 0.5278353632017959}
{'classifier': 'mlp', 'accuracy': 0.6091703056768559, 'F1 (macro)': 0.5830767970211665}


In [29]:
%%time

results = []

from joblib import Parallel, delayed
from sklearn.base import clone

def _one_repeat_f1(X, y, clf, n_splits, seed):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    # clone to avoid any state carryover
    est = clone(clf)
    y_pred = cross_val_predict(est, X, y, cv=cv, n_jobs=1)  # <- avoid nested parallelism
    return f1_score(y, y_pred, average="macro")

def eval_model(X, y, clf, n_splits=5, repeats=10, n_jobs=-1, verbose=0):
    f1s = Parallel(n_jobs=n_jobs, prefer="processes", verbose=verbose)(
        delayed(_one_repeat_f1)(X, y, clf, n_splits, seed)
        for seed in range(repeats)
    )
    return {
        "f1_avg": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s)),
        "f1_max": float(np.max(f1s)),
    }

# ─────────────────────────────── Baseline loop ────────────────────────────────
for emb_col in ["cooc_vector", "cooc_conc_vector", "svd400", "svd400_conc"]:
    context = "concordance" if "conc" in emb_col else "sentence"

    results_dict = {
        "variable_name": emb_col,
        "model": "baseline",
        "context": context
    }

    X = np.stack(load_embedding(emb_col))

    for clf_name, clf in clfs.items():
        scores = eval_model(X, y, clf)
        results_dict[f"{clf_name}_f1_avg"] = scores["f1_avg"]
        results_dict[f"{clf_name}_f1_std"] = scores["f1_std"]
        results_dict[f"{clf_name}_f1_max"] = scores["f1_max"]
    print(results_dict)
    results.append(results_dict)

{'variable_name': 'cooc_vector', 'model': 'baseline', 'context': 'sentence', 'lr_f1_avg': 0.5961386908070991, 'lr_f1_std': 0.010776777113490461, 'lr_f1_max': 0.6153946889072378, 'rf_f1_avg': 0.5825666970957568, 'rf_f1_std': 0.011019845306770599, 'rf_f1_max': 0.5950043932099097, 'hgbc_f1_avg': 0.45299754884995613, 'hgbc_f1_std': 0.015910040527281618, 'hgbc_f1_max': 0.47454722946763855, 'mlp_f1_avg': 0.6010459556143724, 'mlp_f1_std': 0.013480851962075423, 'mlp_f1_max': 0.6206063976259092}
{'variable_name': 'cooc_conc_vector', 'model': 'baseline', 'context': 'concordance', 'lr_f1_avg': 0.6102999524843166, 'lr_f1_std': 0.010049884626001943, 'lr_f1_max': 0.6227982140859543, 'rf_f1_avg': 0.5780550166461695, 'rf_f1_std': 0.0070892391192036676, 'rf_f1_max': 0.584910855258353, 'hgbc_f1_avg': 0.34261345582865577, 'hgbc_f1_std': 0.012533225017800737, 'hgbc_f1_max': 0.3576079642629231, 'mlp_f1_avg': 0.6175801558212617, 'mlp_f1_std': 0.011743839626386792, 'mlp_f1_max': 0.6343573808142294}
{'variabl

In [37]:
results_df = pd.DataFrame(results)
# Define the relevant F1 average columns
f1_cols = ["lr_f1_avg", "rf_f1_avg", "hgbc_f1_avg", "mlp_f1_avg"]
# Determine best model name (column with max F1) for each row
results_df["best_model"] = results_df[f1_cols].idxmax(axis=1).str.replace("_f1_avg", "")
results_df["best_model_f1_performance"] = results_df.apply(
    lambda row: row[f"{row['best_model']}_f1_avg"], axis=1)
results_df.to_csv("../data/labyrinthus_classification_results.csv", index=False)

In [ ]:
for model in ["xlmr", "labert", "labert_wsd"]:
    for context in ["sentence", "concordance"]:
        for L in range(7, 12):
            if model == "labert_wsd" and L < 9:
                continue
            else:
                emb_col = f"embed_l{L}_{model}" if context == "sentence" else f"embed_conc_l{L}_{model}"
                results_dict = {
                    "variable_name": emb_col,
                    "model": model,
                    "emb_type": "contextual",
                    "context": context,
                    "layer": L
                }
                X = np.stack(load_embedding(emb_col))
                for clf_name, clf in clfs.items():
                    scores = eval_model(X, y, clf)
                    results_dict[f"{clf_name}_f1_avg"] = scores["f1_avg"]
                    results_dict[f"{clf_name}_f1_std"] = scores["f1_std"]
                    results_dict[f"{clf_name}_f1_max"] = scores["f1_max"]
                results.append(results_dict)

results_df = pd.DataFrame(results)
# Define the relevant F1 average columns
f1_cols = ["lr_f1_avg", "rf_f1_avg", "hgbc_f1_avg", "mlp_f1_avg"]
# Determine best model name (column with max F1) for each row
results_df["best_model"] = results_df[f1_cols].idxmax(axis=1).str.replace("_f1_avg", "")
results_df["best_model_f1_performance"] = results_df.apply(
    lambda row: row[f"{row['best_model']}_f1_avg"], axis=1)
results_df.to_csv("../data/labyrinthus_classification_results.csv", index=False)

In [ ]:
for model in ["xlmr", "labert", "labert_wsd"]:
    for context in ["sentence", "concordance"]:
        for L in range(7, 12):
            if model == "labert_wsd" and L < 9:
                continue
            else:
                emb_col = f"att_l{L}_{model}_raw_embed" if context == "sentence" else f"att_conc_l{L}_{model}_raw_embed"
                results_dict = {
                    "variable_name": emb_col,
                    "model": model,
                    "emb_type": "attention_raw",
                    "context": context,
                    "layer": L
                }
                X = np.stack(load_embedding(emb_col))
                for clf_name, clf in clfs.items():
                    scores = eval_model(X, y, clf)
                    results_dict[f"{clf_name}_f1_avg"] = scores["f1_avg"]
                    results_dict[f"{clf_name}_f1_std"] = scores["f1_std"]
                    results_dict[f"{clf_name}_f1_max"] = scores["f1_max"]
                results.append(results_dict)

results_df = pd.DataFrame(results)
# Define the relevant F1 average columns
f1_cols = ["lr_f1_avg", "rf_f1_avg", "hgbc_f1_avg", "mlp_f1_avg"]
# Determine best model name (column with max F1) for each row
results_df["best_model"] = results_df[f1_cols].idxmax(axis=1).str.replace("_f1_avg", "")
results_df["best_model_f1_performance"] = results_df.apply(
    lambda row: row[f"{row['best_model']}_f1_avg"], axis=1)
results_df.to_csv("../data/labyrinthus_classification_results.csv", index=False)

In [ ]:
for model in ["xlmr", "labert", "labert_wsd"]:
    for context in ["sentence", "concordance"]:
        for L in range(7, 12):
            if model == "labert_wsd" and L < 9:
                continue
            else:
                emb_col = f"att_l{L}_{model}_svd400_embed" if context == "sentence" else f"att_conc_l{L}_{model}_svd400_embed"
                results_dict = {
                    "variable_name": emb_col,
                    "model": model,
                    "emb_type": "attention_svd",
                    "context": context,
                    "layer": L
                }
                X = np.stack(load_embedding(emb_col))
                for clf_name, clf in clfs.items():
                    scores = eval_model(X, y, clf)
                    results_dict[f"{clf_name}_f1_avg"] = scores["f1_avg"]
                    results_dict[f"{clf_name}_f1_std"] = scores["f1_std"]
                    results_dict[f"{clf_name}_f1_max"] = scores["f1_max"]
                results.append(results_dict)

results_df = pd.DataFrame(results)
# Define the relevant F1 average columns
f1_cols = ["lr_f1_avg", "rf_f1_avg", "hgbc_f1_avg", "mlp_f1_avg"]
# Determine best model name (column with max F1) for each row
results_df["best_model"] = results_df[f1_cols].idxmax(axis=1).str.replace("_f1_avg", "")
results_df["best_model_f1_performance"] = results_df.apply(
    lambda row: row[f"{row['best_model']}_f1_avg"], axis=1)
results_df.to_csv("../data/labyrinthus_classification_results.csv", index=False)

In [ ]:
for model in ["xlmr", "labert", "labert_wsd"]:
    for context in ["sentence", "concordance"]:
        for L in range(7, 12):
            if model == "labert_wsd" and L < 9:
                continue
            else:
                emb1 = f"embed_l{L}_{model}" if context == "sentence" else f"embed_conc_l{L}_{model}"
                emb2 = f"att_l{L}_{model}_raw_embed" if context == "sentence" else f"att_conc_l{L}_{model}_raw_embed"
                results_dict = {
                    "variable_name": emb1 + "&" + emb2,
                    "model": model,
                    "emb_type": "concatenated_raw_attn",
                    "context": context,
                    "layer": L
                }
                X = merge_embeddings([emb1, emb2], method="concat")
                for clf_name, clf in clfs.items():
                    scores = eval_model(X, y, clf)
                    results_dict[f"{clf_name}_f1_avg"] = scores["f1_avg"]
                    results_dict[f"{clf_name}_f1_std"] = scores["f1_std"]
                results_dict[f"{clf_name}_f1_max"] = scores["f1_max"]
            results.append(results_dict)

results_df = pd.DataFrame(results)
# Define the relevant F1 average columns
f1_cols = ["lr_f1_avg", "rf_f1_avg", "hgbc_f1_avg", "mlp_f1_avg"]
# Determine best model name (column with max F1) for each row
results_df["best_model"] = results_df[f1_cols].idxmax(axis=1).str.replace("_f1_avg", "")
results_df["best_model_f1_performance"] = results_df.apply(
    lambda row: row[f"{row['best_model']}_f1_avg"], axis=1)
results_df.to_csv("../data/labyrinthus_classification_results.csv", index=False)

In [ ]:
for model in ["xlmr", "labert", "labert_wsd"]:
    for context in ["sentence", "concordance"]:
        for L in range(7, 12):
            if model == "labert_wsd" and L < 9:
                continue
            else:
                emb1 = f"embed_l{L}_{model}" if context == "sentence" else f"embed_conc_l{L}_{model}"
                emb2 = f"att_l{L}_{model}_svd400_embed" if context == "sentence" else f"att_conc_l{L}_{model}_svd400_embed"
                results_dict = {
                    "variable_name": emb1 + "&" + emb2,
                    "model": model,
                    "emb_type": "concatenated",
                    "context": context,
                    "layer": L
                }
                X = merge_embeddings([emb1, emb2], method="concat")
                for clf_name, clf in clfs.items():
                    scores = eval_model(X, y, clf)
                    results_dict[f"{clf_name}_f1_avg"] = scores["f1_avg"]
                    results_dict[f"{clf_name}_f1_std"] = scores["f1_std"]
                    results_dict[f"{clf_name}_f1_max"] = scores["f1_max"]
                results.append(results_dict)

results_df = pd.DataFrame(results)
# Define the relevant F1 average columns
f1_cols = ["lr_f1_avg", "rf_f1_avg", "hgbc_f1_avg", "mlp_f1_avg"]
# Determine best model name (column with max F1) for each row
results_df["best_model"] = results_df[f1_cols].idxmax(axis=1).str.replace("_f1_avg", "")
results_df["best_model_f1_performance"] = results_df.apply(
    lambda row: row[f"{row['best_model']}_f1_avg"], axis=1)
results_df.to_csv("../data/labyrinthus_classification_results.csv", index=False)

In [ ]:
for model in ["xlmr", "labert", "labert_wsd"]:
    for context in ["sentence", "concordance"]:
        # Pick corresponding baseline column based on context
        baseline_col = "cooc_vector" if context == "sentence" else "cooc_conc_vector"

        for L in range(7, 12):
            if model == "labert_wsd" and L < 9:
                continue
            else:
                emb1 = f"embed_l{L}_{model}" if context == "sentence" else f"embed_conc_l{L}_{model}"
                emb2 = f"att_l{L}_{model}_raw_embed" if context == "sentence" else f"att_conc_l{L}_{model}_raw_embed"
                emb3 = baseline_col  # same across layers

                results_dict = {
                    "variable_name": f"{emb1}&{emb2}&{emb3}",
                    "model": model,
                    "emb_type": "concatenated_raw+baseline",
                    "context": context,
                    "layer": L
                }

                X = merge_embeddings([emb1, emb2, emb3], method="concat")
                for clf_name, clf in clfs.items():
                    scores = eval_model(X, y, clf)
                    results_dict[f"{clf_name}_f1_avg"] = scores["f1_avg"]
                    results_dict[f"{clf_name}_f1_std"] = scores["f1_std"]
                    results_dict[f"{clf_name}_f1_max"] = scores["f1_max"]
                results.append(results_dict)

results_df = pd.DataFrame(results)
# Define the relevant F1 average columns
f1_cols = ["lr_f1_avg", "rf_f1_avg", "hgbc_f1_avg", "mlp_f1_avg"]
# Determine best model name (column with max F1) for each row
results_df["best_model"] = results_df[f1_cols].idxmax(axis=1).str.replace("_f1_avg", "")
results_df["best_model_f1_performance"] = results_df.apply(
    lambda row: row[f"{row['best_model']}_f1_avg"], axis=1)
results_df.to_csv("../data/labyrinthus_classification_results.csv", index=False)

In [ ]:
for model in ["xlmr", "labert", "labert_wsd"]:
    for context in ["sentence", "concordance"]:
        # Pick corresponding baseline column based on context
        baseline_col = "cooc_vector" if context == "sentence" else "cooc_conc_vector"

        for L in range(7, 12):
            if model == "labert_wsd" and L < 9:
                continue
            else:
                emb1 = f"embed_l{L}_{model}" if context == "sentence" else f"embed_conc_l{L}_{model}"
                emb2 = f"att_l{L}_{model}_svd400_embed" if context == "sentence" else f"att_conc_l{L}_{model}_svd400_embed"
                emb3 = baseline_col  # same across layers

                results_dict = {
                    "variable_name": f"{emb1}&{emb2}&{emb3}",
                    "model": model,
                    "emb_type": "concatenated_raw+baseline",
                    "context": context,
                    "layer": L
                }

                X = merge_embeddings([emb1, emb2, emb3], method="concat")
                for clf_name, clf in clfs.items():
                    scores = eval_model(X, y, clf)
                    results_dict[f"{clf_name}_f1_avg"] = scores["f1_avg"]
                    results_dict[f"{clf_name}_f1_std"] = scores["f1_std"]
                    results_dict[f"{clf_name}_f1_max"] = scores["f1_max"]
                results.append(results_dict)

results_df = pd.DataFrame(results)
# Define the relevant F1 average columns
f1_cols = ["lr_f1_avg", "rf_f1_avg", "hgbc_f1_avg", "mlp_f1_avg"]
# Determine best model name (column with max F1) for each row
results_df["best_model"] = results_df[f1_cols].idxmax(axis=1).str.replace("_f1_avg", "")
results_df["best_model_f1_performance"] = results_df.apply(
    lambda row: row[f"{row['best_model']}_f1_avg"], axis=1)
results_df.to_csv("../data/labyrinthus_classification_results.csv", index=False)

In [ ]:
results_df.to_parquet("../data/labyrinthus_classification_results.parquet")